In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.naive_bayes import ComplementNB
from sklearn.metrics import accuracy_score, f1_score, classification_report

In [3]:
import pandas as pd
from google.colab import files
from sklearn.model_selection import train_test_split

# Upload:
# processed_icd10_reference_valid_only.csv
uploaded = files.upload()

print("Uploaded files:")
for name in uploaded.keys():
    print(name)

icd10_file = None

for name in uploaded.keys():
    lower = name.lower()

    if "processed_icd10_reference_valid_only" in lower or "icd10" in lower:
        icd10_file = name

print("Detected ICD-10 file:", icd10_file)

if icd10_file is None:
    raise ValueError("Could not detect the processed ICD-10 file. Please upload processed_icd10_reference_valid_only.csv")

icd10_df = pd.read_csv(icd10_file)

print("icd10_df shape:", icd10_df.shape)
print("Columns:", icd10_df.columns.tolist())
display(icd10_df.head())

Saving processed_icd10_reference_valid_only (1).csv to processed_icd10_reference_valid_only (1).csv
Uploaded files:
processed_icd10_reference_valid_only (1).csv
Detected ICD-10 file: processed_icd10_reference_valid_only (1).csv
icd10_df shape: (9943, 10)
Columns: ['Code', 'Literal', 'Code_clean', 'Literal_basic', 'in_icd_reference', 'D_P', 'Description', 'y_category', 'y_full_code', 'Literal_match']


,Code,Literal,Code_clean,Literal_basic,in_icd_reference,D_P,Description,y_category,y_full_code,Literal_match
0,J9809,Hiperreactividad bronquial,J9809,Hiperreactividad bronquial,True,D,"Otras enfermedades de los bronquios, no clasif...",J,J9809,hiperreactividad bronquial
1,J9801,broncoespástica,J9801,broncoespástica,True,D,Broncoespasmo agudo,J,J9801,broncoespastica
2,I420,miocardiopatía dilatada,I420,miocardiopatía dilatada,True,D,Miocardiopatía dilatada,I,I420,miocardiopatia dilatada
3,Y831,HTA irc 6,Y831,HTA irc 6,True,D,Cirugía con implante de dispositivo interno ar...,Y,Y831,hta irc 6
4,R5600,Crisis febriles atípicas,R5600,Crisis febriles atípicas,True,D,Convulsiones febriles simples,R,R5600,crisis febriles atipicas


In [4]:
# ============================================================
# Notebook 03 setup: upload processed ICD-10 file and create split
# ============================================================

import pandas as pd
import numpy as np
from google.colab import files
from sklearn.model_selection import train_test_split

# Upload:
# processed_icd10_reference_valid_only.csv
uploaded = files.upload()

icd10_file = None

for name in uploaded.keys():
    lower = name.lower()
    if "processed_icd10_reference_valid_only" in lower or "icd10" in lower:
        icd10_file = name

print("Detected ICD-10 file:", icd10_file)

if icd10_file is None:
    raise ValueError("Please upload processed_icd10_reference_valid_only.csv")

icd10_df = pd.read_csv(icd10_file)

print("icd10_df shape:", icd10_df.shape)
print("Columns:", icd10_df.columns.tolist())
display(icd10_df.head())

# Safety checks
required_cols = ["Literal_match", "y_category"]
missing_cols = [c for c in required_cols if c not in icd10_df.columns]

if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

# Clean rows
icd10_df = icd10_df.dropna(subset=["Literal_match", "y_category"]).copy()
icd10_df = icd10_df[icd10_df["Literal_match"].astype(str).str.len() > 0].copy()
icd10_df = icd10_df[icd10_df["y_category"].astype(str).str.len() > 0].copy()

# Stratified split, keeping ultra-rare classes in train
category_counts = icd10_df["y_category"].value_counts()
rare_categories = category_counts[category_counts < 2].index

rare_df = icd10_df[icd10_df["y_category"].isin(rare_categories)].copy()
normal_df = icd10_df[~icd10_df["y_category"].isin(rare_categories)].copy()

train_normal, val_part = train_test_split(
    normal_df,
    test_size=0.2,
    random_state=42,
    stratify=normal_df["y_category"]
)

train_part = pd.concat([train_normal, rare_df], ignore_index=True)

# Create model variables
X_train = train_part["Literal_match"].fillna("").astype(str)
y_train = train_part["y_category"].astype(str)

X_val = val_part["Literal_match"].fillna("").astype(str)
y_val = val_part["y_category"].astype(str)

print("Train shape:", train_part.shape)
print("Validation shape:", val_part.shape)
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_val:", X_val.shape)
print("y_val:", y_val.shape)
print("Train categories:", y_train.nunique())
print("Validation categories:", y_val.nunique())

Saving processed_icd10_reference_valid_only (1).csv to processed_icd10_reference_valid_only (1) (1).csv
Detected ICD-10 file: processed_icd10_reference_valid_only (1) (1).csv
icd10_df shape: (9943, 10)
Columns: ['Code', 'Literal', 'Code_clean', 'Literal_basic', 'in_icd_reference', 'D_P', 'Description', 'y_category', 'y_full_code', 'Literal_match']


,Code,Literal,Code_clean,Literal_basic,in_icd_reference,D_P,Description,y_category,y_full_code,Literal_match
0,J9809,Hiperreactividad bronquial,J9809,Hiperreactividad bronquial,True,D,"Otras enfermedades de los bronquios, no clasif...",J,J9809,hiperreactividad bronquial
1,J9801,broncoespástica,J9801,broncoespástica,True,D,Broncoespasmo agudo,J,J9801,broncoespastica
2,I420,miocardiopatía dilatada,I420,miocardiopatía dilatada,True,D,Miocardiopatía dilatada,I,I420,miocardiopatia dilatada
3,Y831,HTA irc 6,Y831,HTA irc 6,True,D,Cirugía con implante de dispositivo interno ar...,Y,Y831,hta irc 6
4,R5600,Crisis febriles atípicas,R5600,Crisis febriles atípicas,True,D,Convulsiones febriles simples,R,R5600,crisis febriles atipicas


Train shape: (7954, 10)
Validation shape: (1989, 10)
X_train: (7954,)
y_train: (7954,)
X_val: (1989,)
y_val: (1989,)
Train categories: 34
Validation categories: 33


In [5]:
def evaluate_model(name, model):
    model.fit(X_train, y_train)
    pred = model.predict(X_val)

    acc = accuracy_score(y_val, pred)
    macro_f1 = f1_score(y_val, pred, average="macro")
    weighted_f1 = f1_score(y_val, pred, average="weighted")

    print(name)
    print("Accuracy:", acc)
    print("Macro F1:", macro_f1)
    print("Weighted F1:", weighted_f1)

    return {
        "method": name,
        "accuracy": acc,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1
    }, pred, model

In [ ]:
logreg_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(3, 5),
        min_df=1,
        lowercase=True
    )),
    ("clf", LogisticRegression(
        max_iter=3000,
        class_weight=None,
        n_jobs=-1
    ))
])

logreg_result, logreg_pred, logreg_model = evaluate_model(
    "tfidf_char_logreg_category",
    logreg_model
)

tfidf_char_logreg_category
Accuracy: 0.6963298139768728
Macro F1: 0.4847604554690191
Weighted F1: 0.6843359241978458


In [ ]:
cnb_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(3, 5),
        min_df=1,
        lowercase=True
    )),
    ("clf", ComplementNB())
])

cnb_result, cnb_pred, cnb_model = evaluate_model(
    "tfidf_char_complement_nb",
    cnb_model
)

tfidf_char_complement_nb
Accuracy: 0.6817496229260935
Macro F1: 0.5841584175795653
Weighted F1: 0.6769095591218988


In [ ]:
sgd_model = Pipeline([
    ("features", FeatureUnion([
        ("char", TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(2, 5),
            min_df=2,
            sublinear_tf=True,
            lowercase=True
        )),
        ("word", TfidfVectorizer(
            analyzer="word",
            ngram_range=(1, 2),
            min_df=2,
            sublinear_tf=True,
            lowercase=True
        ))
    ])),
    ("clf", SGDClassifier(
        loss="hinge",
        alpha=1e-4,
        max_iter=2000,
        random_state=42
    ))
])

sgd_result, sgd_pred, sgd_model = evaluate_model(
    "char_word_sgdclassifier",
    sgd_model
)

char_word_sgdclassifier
Accuracy: 0.7481146304675717
Macro F1: 0.6451895640946673
Weighted F1: 0.7451278447819131


In [ ]:
svc_model = Pipeline([
    ("features", FeatureUnion([
        ("char", TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(2, 5),
            min_df=2,
            sublinear_tf=True,
            lowercase=True
        )),
        ("word", TfidfVectorizer(
            analyzer="word",
            ngram_range=(1, 2),
            min_df=2,
            sublinear_tf=True,
            lowercase=True
        ))
    ])),
    ("clf", LinearSVC(
        C=0.5,
        class_weight=None,
        max_iter=20000,
        random_state=42
    ))
])

svc_result, svc_pred, svc_model = evaluate_model(
    "char_word_linear_svc_C05",
    svc_model
)

val_part["base_svc_pred"] = svc_pred

char_word_linear_svc_C05
Accuracy: 0.7596782302664655
Macro F1: 0.6582678086693827
Weighted F1: 0.757519953040085


In [ ]:
svc_balanced_model = Pipeline([
    ("features", FeatureUnion([
        ("char", TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(2, 5),
            min_df=2,
            sublinear_tf=True,
            lowercase=True
        )),
        ("word", TfidfVectorizer(
            analyzer="word",
            ngram_range=(1, 2),
            min_df=2,
            sublinear_tf=True,
            lowercase=True
        ))
    ])),
    ("clf", LinearSVC(
        C=0.5,
        class_weight="balanced",
        max_iter=20000,
        random_state=42
    ))
])

svc_bal_result, svc_bal_pred, svc_bal_model = evaluate_model(
    "char_word_linear_svc_balanced",
    svc_balanced_model
)

char_word_linear_svc_balanced
Accuracy: 0.7345399698340875
Macro F1: 0.6521002245842701
Weighted F1: 0.7358529698564548


In [ ]:
# ============================================================
# SGD + SVC selective ensemble
# Use SGD only for categories where SGD beats SVC on validation
# ============================================================

import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Make sure these exist:
# y_val
# svc_pred
# sgd_pred

val_compare = val_part.copy()
val_compare["true_category"] = y_val.values
val_compare["svc_pred"] = svc_pred
val_compare["sgd_pred"] = sgd_pred

val_compare["svc_correct"] = (
    val_compare["svc_pred"].astype(str) == val_compare["true_category"].astype(str)
)

val_compare["sgd_correct"] = (
    val_compare["sgd_pred"].astype(str) == val_compare["true_category"].astype(str)
)

# Category-level performance comparison
category_perf = (
    val_compare
    .groupby("true_category")
    .agg(
        rows=("true_category", "count"),
        svc_accuracy=("svc_correct", "mean"),
        sgd_accuracy=("sgd_correct", "mean")
    )
    .reset_index()
)

category_perf["sgd_minus_svc"] = (
    category_perf["sgd_accuracy"] - category_perf["svc_accuracy"]
)

display(category_perf.sort_values("sgd_minus_svc", ascending=False))

,true_category,rows,svc_accuracy,sgd_accuracy,sgd_minus_svc
1,1,26,0.730769,0.807692,0.076923
17,K,76,0.750000,0.789474,0.039474
3,3,68,0.823529,0.852941,0.029412
13,G,43,0.558140,0.581395,0.023256
32,Z,342,0.839181,0.845029,0.005848
27,U,4,0.250000,0.250000,0.000000
6,8,2,1.000000,1.000000,0.000000
5,5,3,0.333333,0.333333,0.000000
2,2,1,0.000000,0.000000,0.000000
8,B,116,0.896552,0.896552,0.000000


In [ ]:
# ============================================================
# Select categories where SGD performs better than SVC
# ============================================================

# Conservative rule:
# Use SGD only when it beats SVC by at least 0.02
# and the category has enough validation rows
sgd_better_categories = category_perf[
    (category_perf["sgd_minus_svc"] > 0.02) &
    (category_perf["rows"] >= 10)
]["true_category"].astype(str).tolist()

print("Categories where SGD is better:", sgd_better_categories)

# Ensemble prediction:
# If SVC predicts one of the categories where SGD is stronger,
# use SGD prediction. Otherwise keep SVC.
val_compare["sgd_svc_ensemble_pred"] = np.where(
    val_compare["svc_pred"].astype(str).isin(sgd_better_categories),
    val_compare["sgd_pred"].astype(str),
    val_compare["svc_pred"].astype(str)
)

ensemble_acc = accuracy_score(
    val_compare["true_category"].astype(str),
    val_compare["sgd_svc_ensemble_pred"].astype(str)
)

ensemble_macro_f1 = f1_score(
    val_compare["true_category"].astype(str),
    val_compare["sgd_svc_ensemble_pred"].astype(str),
    average="macro"
)

ensemble_weighted_f1 = f1_score(
    val_compare["true_category"].astype(str),
    val_compare["sgd_svc_ensemble_pred"].astype(str),
    average="weighted"
)

print("SVC accuracy:", accuracy_score(y_val, svc_pred))
print("SGD accuracy:", accuracy_score(y_val, sgd_pred))
print("SGD + SVC ensemble accuracy:", ensemble_acc)
print("Ensemble macro F1:", ensemble_macro_f1)
print("Ensemble weighted F1:", ensemble_weighted_f1)

print(classification_report(
    val_compare["true_category"].astype(str),
    val_compare["sgd_svc_ensemble_pred"].astype(str),
    zero_division=0
))

Categories where SGD is better: ['1', '3', 'G', 'K']
SVC accuracy: 0.7596782302664655
SGD accuracy: 0.7481146304675717
SGD + SVC ensemble accuracy: 0.759175465057818
Ensemble macro F1: 0.6573421024038008
Ensemble weighted F1: 0.7570059474419366
              precision    recall  f1-score   support

           0       0.82      0.81      0.82       209
           1       0.79      0.73      0.76        26
           2       0.00      0.00      0.00         1
           3       0.84      0.82      0.83        68
           4       0.67      0.57      0.62         7
           5       1.00      0.33      0.50         3
           8       1.00      1.00      1.00         2
           A       0.00      0.00      0.00         4
           B       0.90      0.90      0.90       116
           C       0.94      0.83      0.88        41
           D       0.77      0.69      0.72        67
           E       0.72      0.74      0.73        87
           F       0.75      0.64      0.69        6

In [ ]:
#A selective SGD + SVC ensemble was tested by using SGD only for categories where it outperformed SVC on validation. Although this improved validation slightly, it did not improve Kaggle performance, suggesting that the category-level switching rule overfitted the validation split.

In [6]:
#fine tuning svc
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
import pandas as pd
import numpy as np

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

configs = []

for C in [0.25, 0.5, 0.75, 1.0]:
    for char_ngram in [(2, 5), (3, 5), (2, 6)]:
        for word_ngram in [(1, 2), (1, 3)]:
            for min_df in [1, 2, 3]:
                configs.append({
                    "C": C,
                    "char_ngram": char_ngram,
                    "word_ngram": word_ngram,
                    "min_df": min_df
                })

print("Number of configs:", len(configs))

tuning_results = []

for cfg in configs:
    model = Pipeline([
        ("features", FeatureUnion([
            ("char", TfidfVectorizer(
                analyzer="char_wb",
                ngram_range=cfg["char_ngram"],
                min_df=cfg["min_df"],
                sublinear_tf=True,
                lowercase=True
            )),
            ("word", TfidfVectorizer(
                analyzer="word",
                ngram_range=cfg["word_ngram"],
                min_df=cfg["min_df"],
                sublinear_tf=True,
                lowercase=True
            ))
        ])),
        ("clf", LinearSVC(
            C=cfg["C"],
            class_weight=None,
            max_iter=20000,
            random_state=42
        ))
    ])

    scores = cross_val_score(
        model,
        icd10_df["Literal_match"].fillna("").astype(str),
        icd10_df["y_category"].astype(str),
        cv=cv,
        scoring="accuracy",
        n_jobs=-1
    )

    tuning_results.append({
        **cfg,
        "mean_accuracy": scores.mean(),
        "std_accuracy": scores.std()
    })

tuning_results_df = pd.DataFrame(tuning_results)
display(tuning_results_df.sort_values("mean_accuracy", ascending=False).head(20))

tuning_results_df.to_csv("linear_svc_tuning_results.csv", index=False)

Number of configs: 72


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklea

KeyboardInterrupt: 

In [8]:
#testing with abbreviation expansion in dataset

# ============================================================
# Abbreviation expansion experiment
# ============================================================

ABBREV_MAP = {
    "dm": "diabetes mellitus",
    "hta": "hipertension arterial",
    "irc": "insuficiencia renal cronica",
    "epoc": "enfermedad pulmonar obstructiva cronica",
    "iam": "infarto agudo miocardio",
    "acv": "accidente cerebrovascular",
    "vrs": "virus respiratorio sincitial",
    "hpv": "virus papiloma humano",
    "vih": "virus inmunodeficiencia humana",
    "hpb": "hiperplasia prostatica benigna",
    "itu": "infeccion tracto urinario",
    "rge": "reflujo gastroesofagico",
    "erc": "enfermedad renal cronica"
}

def expand_abbreviations(text):
    tokens = str(text).split()
    expanded = []

    for tok in tokens:
        clean_tok = tok.lower().strip()
        if clean_tok in ABBREV_MAP:
            expanded.append(clean_tok)
            expanded.append(ABBREV_MAP[clean_tok])
        else:
            expanded.append(tok)

    return " ".join(expanded)

train_part["Literal_abbrev_expanded"] = train_part["Literal_match"].apply(expand_abbreviations)
val_part["Literal_abbrev_expanded"] = val_part["Literal_match"].apply(expand_abbreviations)

abbrev_model = Pipeline([
    ("features", FeatureUnion([
        ("char", TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(2, 5),
            min_df=2,
            sublinear_tf=True,
            lowercase=True
        )),
        ("word", TfidfVectorizer(
            analyzer="word",
            ngram_range=(1, 2),
            min_df=2,
            sublinear_tf=True,
            lowercase=True
        ))
    ])),
    ("clf", LinearSVC(
        C=0.5,
        class_weight=None,
        max_iter=20000,
        random_state=42
    ))
])

abbrev_model.fit(
    train_part["Literal_abbrev_expanded"].fillna("").astype(str),
    train_part["y_category"].astype(str)
)

val_part["abbrev_svc_pred"] = abbrev_model.predict(
    val_part["Literal_abbrev_expanded"].fillna("").astype(str)
)

abbrev_acc = accuracy_score(
    val_part["y_category"].astype(str),
    val_part["abbrev_svc_pred"].astype(str)
)


print("Abbreviation-expanded SVC accuracy:", abbrev_acc)

Abbreviation-expanded SVC accuracy: 0.7576671694318753


In [9]:
#adding context flags

# ============================================================
# Broad context flag model
# ============================================================

CONTEXT_PATTERNS = {
    "CTX_OBSTETRIC": [
        "parto", "part", "puerperio", "puerperi", "embarazo",
        "gestacion", "gestante", "cesarea", "fetal", "feto"
    ],
    "CTX_NEWBORN": [
        "rn", "recien nacido", "neonato", "prematuro", "prematura"
    ],
    "CTX_RESPIRATORY": [
        "asma", "epoc", "bronquitis", "neumonia", "respiratoria"
    ],
    "CTX_URINARY": [
        "itu", "renal", "riñon", "rinon", "vejiga", "ureter", "prostata"
    ],
    "CTX_INFECTION": [
        "covid", "vih", "vrs", "vhb", "vhc", "infeccion", "sepsis"
    ],
    "CTX_ONCOLOGY": [
        "tumor", "neoplasia", "carcinoma", "adenocarcinoma", "cancer", "ca"
    ],
    "CTX_TRAUMA": [
        "fractura", "trauma", "contusion", "herida", "lesion"
    ],
    "CTX_PROCEDURE": [
        "biopsia", "reseccion", "extirpacion", "colocacion", "retirada",
        "drenaje", "cirugia", "intervencion"
    ]
}

def add_context_flags(text):
    text = str(text).lower()
    flags = []

    for flag, terms in CONTEXT_PATTERNS.items():
        if any(term in text for term in terms):
            flags.append(flag)

    if flags:
        return text + " " + " ".join(flags)

    return text

train_part["Literal_context_flags"] = train_part["Literal_match"].apply(add_context_flags)
val_part["Literal_context_flags"] = val_part["Literal_match"].apply(add_context_flags)

context_flag_model = Pipeline([
    ("features", FeatureUnion([
        ("char", TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(2, 5),
            min_df=2,
            sublinear_tf=True,
            lowercase=True
        )),
        ("word", TfidfVectorizer(
            analyzer="word",
            ngram_range=(1, 2),
            min_df=2,
            sublinear_tf=True,
            lowercase=True
        ))
    ])),
    ("clf", LinearSVC(
        C=0.5,
        class_weight=None,
        max_iter=20000,
        random_state=42
    ))
])

context_flag_model.fit(
    train_part["Literal_context_flags"].fillna("").astype(str),
    train_part["y_category"].astype(str)
)

val_part["context_flag_pred"] = context_flag_model.predict(
    val_part["Literal_context_flags"].fillna("").astype(str)
)

context_flag_acc = accuracy_score(
    val_part["y_category"].astype(str),
    val_part["context_flag_pred"].astype(str)
)


print("Broad context flag SVC accuracy:", context_flag_acc)


Broad context flag SVC accuracy: 0.7576671694318753


In [ ]:
ensemble_result = {
    "method": "sgd_svc_selective_ensemble",
    "accuracy": ensemble_acc,
    "macro_f1": ensemble_macro_f1,
    "weighted_f1": ensemble_weighted_f1
}

model_summary = pd.DataFrame([
    logreg_result,
    cnb_result,
    sgd_result,
    svc_result,
    svc_bal_result,
    ensemble_result
])

display(model_summary.sort_values("accuracy", ascending=False))

model_summary.to_csv("model_experiment_summary.csv", index=False)
val_compare.to_csv("validation_predictions_sgd_svc_ensemble.csv", index=False)

,method,accuracy,macro_f1,weighted_f1
3,char_word_linear_svc_C05,0.759678,0.658268,0.757520
5,sgd_svc_selective_ensemble,0.759175,0.657342,0.757006
2,char_word_sgdclassifier,0.748115,0.645190,0.745128
4,char_word_linear_svc_balanced,0.734540,0.652100,0.735853
0,tfidf_char_logreg_category,0.696330,0.484760,0.684336
1,tfidf_char_complement_nb,0.681750,0.584158,0.676910
